In [ ]:
pip install shap

In [3]:
########################################
# JUPYTER NOTEBOOK MODEL TRAINING CODE #
########################################

# ========================== #
# 1. IMPORT REQUIRED LIBRARIES
# ========================== #
import os
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, medfilt
from scipy import stats
from scipy.stats import entropy, median_abs_deviation
from scipy.fft import fft, fftfreq
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn and Imblearn
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE

# For optional t-SNE visualization
from sklearn.manifold import TSNE

# For optional SHAP analysis
import shap

# Matplotlib settings for inline plotting
%matplotlib inline



In [ ]:
# ======================= #
# 2. DATA PREPROCESSING
# ======================= #

def apply_filter(data, ftype='high', cutoff=10, fs=100.0, order=5):
    """
    Applies a Butterworth filter (default high-pass) to the data.
    """
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    if normal_cutoff >= 1.0:
        return data  # No filtering if cutoff is too high
    b, a = butter(order, normal_cutoff, btype=ftype, analog=False)
    y = filtfilt(b, a, data)
    return y

def remove_spikes(data, threshold=3):
    """
    Removes spikes from the data using a modified Z-score approach.
    Spikes are replaced by the median of the data.
    """
    median_val = np.median(data)
    mad = median_abs_deviation(data)
    if mad == 0:
        mad = 1e-12  # Avoid division by zero
    z_scores = 0.6745 * (data - median_val) / mad
    spike_indices = np.abs(z_scores) > threshold
    data_cleaned = data.copy()
    data_cleaned[spike_indices] = median_val
    return data_cleaned

def calculate_entropy(signal):
    """
    Calculates the Shannon entropy of the signal's amplitude distribution.
    """
    histogram, _ = np.histogram(signal, bins=256, density=True)
    histogram = histogram + 1e-12  # Avoid zeros
    return entropy(histogram)

def calculate_spectral_entropy(signal, fs=100.0):
    """
    Calculates spectral entropy using the Fourier transform of the signal.
    """
    fft_vals = np.abs(fft(signal))
    fft_norm = fft_vals / np.sum(fft_vals)
    fft_norm = fft_norm + 1e-12
    return entropy(fft_norm)

def extract_features(torque, fs=100.0, window_size=300, threshold=3, kernel_size=5):
    """
    Extracts a comprehensive set of time-domain and frequency-domain features
    from the torque data, with spike removal and filtering.
    """
    # Apply high-pass filter
    filtered_torque = apply_filter(torque, ftype='high', cutoff=10, fs=fs, order=5)
    
    # Remove spikes
    torque_cleaned = remove_spikes(filtered_torque, threshold=threshold)
    
    # Median filter
    torque_filtered = medfilt(torque_cleaned, kernel_size=kernel_size)
    
    # Convert to Pandas Series
    torque_series = pd.Series(torque_filtered)
    
    # --- Time-Domain Statistical Features ---
    mean_val = np.mean(torque_filtered)
    median_val = np.median(torque_filtered)
    mad_val = median_abs_deviation(torque_filtered)
    std_val = np.std(torque_filtered)
    rms_val = np.sqrt(np.mean(torque_filtered**2))
    
    max_val = np.max(torque_filtered)
    min_val = np.min(torque_filtered)
    peak = max(abs(max_val), abs(min_val))
    mean_abs = np.mean(np.abs(torque_filtered))
    
    skew_val = stats.skew(torque_filtered)
    kurt_val = stats.kurtosis(torque_filtered, fisher=False)
    
    # Shape factor and Crest factor
    shape_factor = rms_val / mean_abs if mean_abs != 0 else 0
    crest_factor = peak / rms_val if rms_val != 0 else 0
    
    # Entropy
    signal_entropy = calculate_entropy(torque_filtered)
    
    # Gradient features
    gradient = np.gradient(torque_filtered)
    gradient_mean = np.mean(gradient)
    gradient_std = np.std(gradient)
    
    # Rolling statistics
    rolling_median = torque_series.rolling(window_size, min_periods=1).median()
    rolling_mad = torque_series.rolling(window_size, min_periods=1).apply(median_abs_deviation)
    
    # --- Frequency-Domain Features ---
    N = len(torque_filtered)
    freqs = fftfreq(N, 1/fs)
    fft_vals = fft(torque_filtered)
    fft_magnitude = np.abs(fft_vals)
    
    # Keep only positive frequencies
    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    fft_magnitude = fft_magnitude[pos_mask]
    
    # Spectral features
    spectral_centroid = np.sum(freqs * fft_magnitude) / np.sum(fft_magnitude)
    spectral_ent = calculate_spectral_entropy(torque_filtered, fs)
    peak_frequency = freqs[np.argmax(fft_magnitude)]
    
    # Compile features into a dictionary
    features = {
        'Mean': mean_val,
        'Median': median_val,
        'MAD': mad_val,
        'Standard Deviation': std_val,
        'RMS': rms_val,
        'Shape Factor': shape_factor,
        'Crest Factor': crest_factor,
        'Entropy': signal_entropy,
        'Skewness': skew_val,
        'Kurtosis': kurt_val,
        'Gradient Mean': gradient_mean,
        'Gradient Std Dev': gradient_std,
        'Rolling Median Mean': rolling_median.mean(),
        'Rolling MAD Mean': rolling_mad.mean(),
        'Spectral Centroid': spectral_centroid,
        'Spectral Entropy': spectral_ent,
        'Peak Frequency': peak_frequency
    }
    
    return features


def compile_features_from_folder(folder_path, fs=100.0):
    """
    Processes all CSV files in 'passed' and 'failed' subfolders,
    extracts features, and returns a DataFrame with features and labels.
    """
    features_list = []
    labels = []
    
    # Define subfolders and labels
    subfolders = {
        'passed': 1,
        'failed': 0
    }
    
    for subfolder_name, label in subfolders.items():
        subfolder_path = os.path.join(folder_path, subfolder_name)
        if not os.path.isdir(subfolder_path):
            print(f"Subfolder '{subfolder_name}' not found in '{folder_path}'. Skipping.")
            continue
        
        for filename in os.listdir(subfolder_path):
            if filename.endswith('.csv') and not filename.startswith('.'):
                file_path = os.path.join(subfolder_path, filename)
                
                # Read CSV
                try:
                    data = pd.read_csv(file_path)
                    data.columns = data.columns.str.strip()  # remove extra spaces in columns
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
                    continue
                
                if 'N[Ncm]' not in data.columns:
                    print(f"No 'N[Ncm]' column in {file_path}. Skipping.")
                    continue
                
                # Extract torque data
                torque_data = data['N[Ncm]'].values
                
                # Extract features
                feats = extract_features(torque_data, fs=fs)
                
                # Attempt to parse filename for additional metadata (optional)
                parts = filename.replace('.csv', '').split('_')
                datetime_str, size_str, no_str = None, None, None
                if len(parts) >= 3:
                    datetime_str = parts[0]
                    size_str = parts[1]
                    no_str = parts[2]
                
                feats['Datetime'] = datetime_str
                feats['Size'] = size_str
                feats['No'] = no_str
                features_list.append(feats)
                
                labels.append(label)
    
    df = pd.DataFrame(features_list)
    df['Label'] = labels
    return df



In [ ]:
# ============================== #
# 3. LOAD & VISUALIZE DATA (OPTIONAL)
# ============================== #

def visualize_feature_distribution(features_df):
    """
    Plots histograms and pair plots for an overview of feature distributions.
    WARNING: Pair plots can be slow for many features.
    """
    feature_cols = list(features_df.drop(columns=['Label', 'Datetime', 'Size', 'No']).columns)

    # Histograms
    features_df[feature_cols].hist(bins=30, figsize=(15, 10))
    plt.tight_layout()
    plt.show()

    # Pair Plot (comment out if large dataset or many features)
    sns.pairplot(features_df[feature_cols + ['Label']], hue='Label', diag_kind='kde')
    plt.show()



In [ ]:
# ========================= #
# 4. MODEL TRAINING & EVALUATION
# ========================= #

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

def train_and_evaluate_model(folder_path):
    """
    Main pipeline function:
      1. Compiles features.
      2. Visualizes data (optional).
      3. Splits data, applies SMOTE.
      4. Trains multiple classifiers with GridSearchCV.
      5. Evaluates best model.
      6. Returns best model.
    """
    # --- 4.1 Compile features from folder ---
    features_df = compile_features_from_folder(folder_path, fs=100.0)
    print("Sample of extracted features:\n", features_df.head())

    # Optional data visualization
    print("\nVisualizing feature distributions (this may take time)...")
    visualize_feature_distribution(features_df)

    # --- 4.2 Prepare data for training ---
    X = features_df.drop(columns=['Label', 'Datetime', 'Size', 'No'])
    y = features_df['Label']

    # Check for NaNs or infinite values
    if X.isnull().values.any() or np.isinf(X.values).any():
        print("Data contains NaN or infinite values. Cleaning them up...")
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.dropna()
        y = y.loc[X.index]

    # --- 4.3 Train-test split ---
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, stratify=y, random_state=42
    )

    # --- 4.4 Apply SMOTE to handle class imbalance ---
    smote = SMOTE(random_state=42)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    # --- 4.5 Define pipeline and parameter grid ---
    pipeline = Pipeline([
        ('scaler', StandardScaler()),  # scaling
        ('pca', PCA()),                # PCA
        ('classifier', LogisticRegression())  # placeholder classifier
    ])

    param_grid = [
        # Logistic Regression
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
            'classifier__C': [0.01, 0.1, 1, 10],
            'classifier__penalty': ['l2'],
            'classifier__solver': ['lbfgs']
        },
        # Random Forest
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [RandomForestClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth': [None, 10, 20],
            'classifier__min_samples_split': [2, 5]
        },
        # SVC
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [SVC(probability=True, random_state=42)],
            'classifier__C': [0.1, 1, 10],
            'classifier__kernel': ['rbf', 'linear']
        },
        # K-Nearest Neighbors
        {
            'pca__n_components': [5, 10, 15, None],
            'classifier': [KNeighborsClassifier()],
            'classifier__n_neighbors': [3, 5, 7],
            'classifier__weights': ['uniform', 'distance']
        },
        # Gradient Boosting
        {
            'pca__n_components': [5, 10, None],
            'classifier': [GradientBoostingClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100],
            'classifier__learning_rate': [0.01, 0.1],
            'classifier__max_depth': [3, 5]
        },
        # AdaBoost
        {
            'pca__n_components': [5, 10, None],
            'classifier': [AdaBoostClassifier(random_state=42)],
            'classifier__n_estimators': [50, 100],
            'classifier__learning_rate': [0.01, 0.1, 1.0]
        },
        # Decision Tree (Depth 2-3)
        {
            'pca__n_components': [None],  # typically skip PCA for Decision Trees
            'classifier': [DecisionTreeClassifier(random_state=42)],
            'classifier__max_depth': [2, 3],
            'classifier__min_samples_split': [2, 5],
            'classifier__criterion': ['gini', 'entropy']
        }
    ]

    # --- 4.6 Set up cross-validation and GridSearchCV ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        pipeline, param_grid, cv=cv, scoring='accuracy',
        n_jobs=-1, refit='accuracy', return_train_score=True
    )

    # --- 4.7 Fit the model with cross-validation ---
    grid_search.fit(X_train_res, y_train_res)

    # --- 4.8 Display best parameters and score ---
    print(f"\nBest Parameters: {grid_search.best_params_}")
    print(f"Best Cross-Validation Score (Accuracy): {grid_search.best_score_:.2f}")

    best_model = grid_search.best_estimator_

    # --- 4.9 Evaluate on test set ---
    y_pred = best_model.predict(X_test)
    if hasattr(best_model, "predict_proba"):
        y_proba = best_model.predict_proba(X_test)[:, 1]
    else:
        # Fallback for classifiers without predict_proba
        y_proba = y_pred

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=1)
    recall = recall_score(y_test, y_pred, pos_label=1)
    f1 = f1_score(y_test, y_pred, pos_label=1)
    roc_auc = roc_auc_score(y_test, y_proba)

    print("\nTest Set Performance:")
    print(f"Accuracy:  {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1 Score:  {f1:.2f}")
    print(f"ROC AUC:   {roc_auc:.2f}")

    # Classification report
    report = classification_report(y_test, y_pred, target_names=['Failed', 'Passed'], output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report = df_report[['precision', 'recall', 'f1-score', 'support']]
    df_report.columns = ['Precision', 'Recall', 'F1 Score', 'Support']
    df_report = df_report.round(2)
    print("\nClassification Report:\n", df_report)

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Failed', 'Passed'], yticklabels=['Failed', 'Passed'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.show()

    # ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc_val = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc='lower right')
    plt.show()

    # --- 4.10 (OPTIONAL) Visualize Decision Tree if Best Model is a Decision Tree ---
    if isinstance(best_model.named_steps['classifier'], DecisionTreeClassifier):
        tree_clf = best_model.named_steps['classifier']

        # Determine feature names (handle PCA)
        if best_model.named_steps['pca'] is None:
            feature_names = X.columns
        else:
            # If PCA was used but set to None, it means no PCA. Otherwise, create PC names.
            n_pcs = best_model.named_steps['pca'].n_components_
            feature_names = [f'PC{i+1}' for i in range(n_pcs)] if n_pcs else X.columns

        plt.figure(figsize=(18, 10))
        plot_tree(tree_clf, feature_names=feature_names, class_names=['Failed', 'Passed'], filled=True, rounded=True)
        plt.title('Decision Tree Visualization (Depth=2-3)')
        plt.show()

    return best_model, (X_test, y_test)




In [ ]:
# ============================ #
# 5. (OPTIONAL) T-SNE VISUALIZATION
# ============================ #
def visualize_tsne(X, y):
    """
    Uses t-SNE for 2D visualization of the feature space.
    """
    tsne = TSNE(n_components=2, random_state=42)
    X_tsne = tsne.fit_transform(X)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y, palette='viridis')
    plt.title('t-SNE Visualization of Torque Data')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.legend(title='Label', labels=['Failed', 'Passed'])
    plt.show()



In [ ]:
# ============================== #
# 6. (OPTIONAL) SHAP ANALYSIS
# ============================== #
def plot_shap_values(pipeline_model, X_sample):
    """
    Plots SHAP values for tree-based or linear models.
    If the classifier uses PCA, SHAP values may be on principal components (less interpretable).
    """
    classifier = pipeline_model.named_steps['classifier']
    # Identify classifier type
    if isinstance(classifier, (RandomForestClassifier, GradientBoostingClassifier)):
        explainer = shap.TreeExplainer(classifier)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, feature_names=X_sample.columns)
    elif isinstance(classifier, LogisticRegression):
        explainer = shap.LinearExplainer(classifier, X_sample)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, feature_names=X_sample.columns)
    elif isinstance(classifier, SVC):
        # For SVC, we typically use KernelExplainer
        explainer = shap.KernelExplainer(classifier.predict_proba, X_sample)
        shap_values = explainer.shap_values(X_sample)
        shap.summary_plot(shap_values, X_sample, feature_names=X_sample.columns)
    else:
        print("SHAP analysis not directly supported for this classifier.")



In [ ]:

# ============================================== #
# 7. FULL NOTEBOOK EXAMPLE USAGE (MAIN EXECUTION)
# ============================================== #

# Example usage: adjust path to your dataset folder
folder_path = r"C:\Your\Path\To\Data\Folder"  # Replace with your local path

# Train model and get best model + test data
best_model, (X_test, y_test) = train_and_evaluate_model(folder_path)

# (Optional) Visualize data in 2D with t-SNE
# Convert X_test to a scaled DataFrame if needed
# Already scaled in the pipeline? We can do an extra scale for visualization only
"""
scaler_viz = StandardScaler()
X_test_scaled = scaler_viz.fit_transform(X_test)
visualize_tsne(X_test_scaled, y_test)
"""

# (Optional) SHAP analysis on a subset (requires data in DataFrame form)
"""
# If PCA was used in the best_model, the feature space may be principal components
# For interpretability, you might retrain a model without PCA or convert PC's back 
# to the original space (not trivial). For demonstration, let's assume no PCA is used:

X_sample = X_test.iloc[:50]  # small sample for faster SHAP
plot_shap_values(best_model, X_sample)
"""

print("\nDone.")
